[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nishidaab26002-arch/sake-review-analysis/blob/main/programs/06_analysis_visualization.ipynb)

# 06. 分析・可視化

05で学習したモデルから、**銘柄の味わい空間**を可視化・解釈します（入力: Doc2Vecモデル + トークンCSV）。

1. 銘柄ベクトルの取得 → k-means(k=4) でグループ分け / PCAで2次元化
2. 味わい空間マップ
3. 味わいグループの特徴（キーワード出現率 → レーダーチャート）
4. 類似銘柄の検索
5. 新規銘柄を既存の空間に配置
6. 水の硬度との関係（検証）

In [ ]:
# 依存ライブラリのインストール（Colab）
!pip install -q gensim scikit-learn scipy pandas numpy matplotlib japanize-matplotlib sudachipy sudachidict-full

## 設定

In [ ]:
# --- 実行環境の設定 ---------------------------------------------------------
# Colab: Google Driveをマウントし、BASE_DIR 配下を読み書きします。
# ローカル: 環境変数 SAKE_PROJECT_DIR でデータ置き場を指定できます。
import os, sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/作成したプログラム等/日本酒口コミ分析'  # 自分のDriveのパスに変更
    # sake_nlp.py をDriveの BASE_DIR 直下に置いておく。
    # 日本語パスのままだとimportに失敗することがあるため、/contentにコピーして読み込む。
    import shutil, sys
    shutil.copy(f'{BASE_DIR}/sake_nlp.py', '/content/sake_nlp.py')
    sys.path.insert(0, '/content')
except ImportError:
    BASE_DIR = os.environ.get('SAKE_PROJECT_DIR', './sake_review_analysis')

CSV_DIR = f'{BASE_DIR}/口コミ_CSV'          # 収集・加工したCSV
MODEL_DIR = f'{BASE_DIR}/機械学習モデル'      # 学習済みDoc2Vecモデル
for d in (CSV_DIR, MODEL_DIR):
    os.makedirs(d, exist_ok=True)

REVIEWS_CSV = f'{CSV_DIR}/reviews_with_water_hardness.csv'   # 03で作成した口コミ+硬度
TOKENIZED_CSV = f'{CSV_DIR}/reviews_tokenized.csv'           # 04の出力 = 05, 06の入力
MODEL_PATH = f'{MODEL_DIR}/final_masked_sudachi_doc2vec.model'  # 05の出力 = 06の入力

WATER_MASTER_PATH = f'{CSV_DIR}/maker_water_quality_master.csv'   # 蔵元ごとの水の硬度
N_CLUSTERS = 4

## 1. モデル・データの読み込みとクラスタリング

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib
from gensim.models.doc2vec import Doc2Vec
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import sake_nlp as nlp

model = Doc2Vec.load(MODEL_PATH)
df = nlp.load_tokenized(TOKENIZED_CSV)

tags = list(model.dv.index_to_key)
X = np.array([model.dv[t] for t in tags])          # 銘柄 × 75次元
print(f'{X.shape[0]}銘柄 × {X.shape[1]}次元')

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X)              # 75次元のまま分類（PCAは可視化専用）

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)
variance_ratio = pca.explained_variance_ratio_
print('PCA寄与率:', np.round(variance_ratio * 100, 2), '%')

def brand_of(tag):
    parts = tag.split('_')
    return parts[1] if len(parts) == 3 else tag

## 2. 味わい空間マップ

In [ ]:
from scipy.stats import gaussian_kde
from matplotlib.colors import LinearSegmentedColormap

def draw_kde_cloud(ax, points, color, levels=8, alpha_max=0.5):
    """クラスタの広がりを半透明の「雲」(KDE) で描く"""
    if len(points) < 5:
        return
    x, y = points[:, 0], points[:, 1]
    mx, my = (x.max() - x.min()) * 0.3 + 0.5, (y.max() - y.min()) * 0.3 + 0.5
    xx, yy = np.mgrid[x.min() - mx:x.max() + mx:100j, y.min() - my:y.max() + my:100j]
    density = gaussian_kde(np.vstack([x, y]))(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
    cmap = LinearSegmentedColormap.from_list('cloud', [(1, 1, 1, 0), color], N=256)
    ax.contourf(xx, yy, density, levels=levels, cmap=cmap, alpha=alpha_max, zorder=1)
    ax.contour(xx, yy, density, levels=[density.max() * 0.15], colors=[color], linewidths=1.2, alpha=0.6, zorder=1)

HIGHLIGHT = ["片野桜", "秋鹿", "荷札酒", "満寿泉", "玉川", "作", "天吹", "風の森", "みむろ杉", "百楽門",
             "るみ子の酒", "真澄", "鶴齢", "信州亀齢", "獺祭", "栄光冨士", "出羽桜", "紀土", "黒牛", "新政", "産土", "伯楽星"]
cluster_colors = [plt.colormaps.get_cmap('Set1')(i) for i in range(N_CLUSTERS)]

fig, ax = plt.subplots(figsize=(12, 10))
for c in range(N_CLUSTERS):
    pts = X_pca[cluster_labels == c]
    draw_kde_cloud(ax, pts, cluster_colors[c])
    ax.scatter(pts[:, 0], pts[:, 1], alpha=0.7, color=cluster_colors[c], s=60,
               edgecolors='white', linewidth=0.3, zorder=2, label=f'味わいグループ {c + 1}')

for i, tag in enumerate(tags):                       # 代表銘柄にラベル
    if any(kw in brand_of(tag) for kw in HIGHLIGHT):
        ax.scatter(*X_pca[i], color=cluster_colors[cluster_labels[i]], edgecolors='black', s=150, linewidth=1.5, zorder=5)
        ax.text(X_pca[i, 0] + 0.03, X_pca[i, 1] + 0.03, brand_of(tag), fontsize=11, weight='bold', zorder=6)

ax.set_title('Doc2Vec × PCA 日本酒の味わい空間マップ', fontsize=14)
ax.set_xlabel(f'PC1（寄与率 {variance_ratio[0]*100:.2f}%）')
ax.set_ylabel(f'PC2（寄与率 {variance_ratio[1]*100:.2f}%）')
ax.grid(True, linestyle='--', alpha=0.3)
ax.legend(loc='upper right')
plt.tight_layout(); plt.show()

## 3. 味わいグループの特徴（キーワード出現率）

各銘柄について「味わいキーワードを含む口コミの割合」を求め、グループ平均をレーダーチャートで表示します。
キーワードも口コミと同じSudachi正規化形に変換して照合します。

In [ ]:
from collections import defaultdict
from sklearn.preprocessing import MinMaxScaler

keywords_dict = {
    'isoamyl':   ['メロン', 'バナナ', '穏やかな香り', 'フルーティ'],                       # フルーティ（イソアミル系）
    'caproate':  ['リンゴ', 'りんご', '洋梨', '洋なし', 'マスカット', '派手な香り', '華やかな香り', 'フルーティ'],  # フルーティ（カプロン系）
    'sweetness': ['甘み', '甘味', '甘い', '甘口'],
    'acidity':   ['酸味', '酸っぱ', 'ジューシー', 'キレ'],
    'bitterness':['苦味', '苦み', '渋み', '渋味', '複雑'],
    'dryness':   ['辛口', '辛み', 'ドライ', 'シャープ', 'すっきり'],
    'umami':     ['旨味', '旨み', 'うまみ', 'コク', 'まろやか'],
}
keywords_norm = {t: {nlp.normalize_word(w) for w in ws} for t, ws in keywords_dict.items()}

# 銘柄 → 口コミごとのトークン集合
token_sets = defaultdict(list)
for tag, toks in zip(df['tag'], df['tokens']):
    token_sets[tag].append(set(toks))

# 銘柄 × 味わいの「出現率」と「出現件数」
ratio = pd.DataFrame({t: {tag: sum(1 for s in token_sets[tag] if kw & s) / max(len(token_sets[tag]), 1)
                          for tag in tags} for t, kw in keywords_norm.items()})
count = pd.DataFrame({t: {tag: sum(1 for s in token_sets[tag] if kw & s) for tag in tags}
                      for t, kw in keywords_norm.items()})
total_reviews = pd.Series({tag: len(token_sets[tag]) for tag in tags})

# 0〜5スケール化 → グループ平均 → 差が見えやすいよう極値調整(alpha)を加えて表示
scaler = MinMaxScaler(feature_range=(0, 5))
taste_scaled = pd.DataFrame(scaler.fit_transform(ratio), index=ratio.index, columns=ratio.columns)

group_mean = taste_scaled.groupby(cluster_labels).mean()
group_mean_extreme = pd.DataFrame(scaler.fit_transform(group_mean), index=group_mean.index, columns=group_mean.columns)
ALPHA = 0.3
radar_scores = group_mean * (1 - ALPHA) + group_mean_extreme * ALPHA

In [ ]:
labels = list(keywords_dict.keys())
angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
angles += angles[:1]

fig, axes = plt.subplots(2, 3, figsize=(15, 12), subplot_kw={'projection': 'polar'})
axes = axes.flatten()
for ax, (c, row) in zip(axes, radar_scores.iterrows()):
    vals = row.tolist() + row.tolist()[:1]
    ax.plot(angles, vals, linewidth=2); ax.fill(angles, vals, alpha=0.25)
    ax.set_xticks(angles[:-1]); ax.set_xticklabels(labels, fontsize=12)
    ax.set_ylim(0, 5); ax.grid(True)
    ax.set_title(f'味わいグループ {c + 1}（{int((cluster_labels == c).sum())}銘柄）', va='bottom', fontsize=14)
for ax in axes[len(radar_scores):]:
    fig.delaxes(ax)
plt.suptitle('各味わいグループのレーダーチャート', fontsize=18, y=1.02)
plt.tight_layout(); plt.show()

summary = pd.DataFrame({'銘柄数': pd.Series(cluster_labels).value_counts().sort_index(),
                        '総口コミ件数': total_reviews.groupby(cluster_labels).sum()})
summary.index = [f'グループ {i + 1}' for i in summary.index]
display(summary)
display((ratio.groupby(cluster_labels).mean() * 100).round(1).add_suffix('(%)'))

## 4. 類似銘柄の検索

銘柄ベクトル間のコサイン類似度で、味わいが近い銘柄を探します。

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def similar_brands(brand_name, topn=5):
    target = next((t for t in tags if brand_name in brand_of(t)), None)
    if target is None:
        print(f'{brand_name} は見つかりません'); return
    sims = cosine_similarity([model.dv[target]], X)[0]
    print(f'【{brand_name}】に近い銘柄:')
    for i in np.argsort(sims)[::-1][1:topn + 1]:
        print(f'  - {brand_of(tags[i])} ({sims[i]:.3f})')

similar_brands('十四代')

## 5. 新規銘柄を既存の空間に配置する

学習に使っていない銘柄の口コミから `infer_vector` でベクトルを推論し、**学習済みの PCA / k-means にそのまま当てはめ**ます（再学習はしません）。
前処理は学習時と同じ `sake_nlp` の関数を使います。

In [ ]:
NEW_REVIEW_CSV = f'{CSV_DIR}/saketime_分類テスト用_analyzed.csv'   # 列: prefecture, brand, maker, review_body
INFER_EPOCHS = 20
N_INFER_RUNS = 5          # infer_vectorは乱数を含むため複数回の平均をとる

df_new = pd.read_csv(NEW_REVIEW_CSV, encoding='utf-8-sig').dropna(subset=['review_body'])
df_new['tag'] = df_new['prefecture'] + '_' + df_new['brand'] + '_' + df_new['maker']

new_tokens = defaultdict(list)
for tag, text in zip(df_new['tag'], df_new['review_body']):
    toks = nlp.tokenize(nlp.mask_review(text))
    if len(toks) >= 3:
        new_tokens[tag].extend(toks)

new_tags = list(new_tokens)
X_new = np.array([np.mean([model.infer_vector(new_tokens[t], epochs=INFER_EPOCHS) for _ in range(N_INFER_RUNS)], axis=0)
                  for t in new_tags])
X_new_pca = pca.transform(X_new)            # 再fitせず transform のみ
new_clusters = kmeans.predict(X_new)        # 再fitせず predict のみ

display(pd.DataFrame({'tag': new_tags, 'PC1': X_new_pca[:, 0], 'PC2': X_new_pca[:, 1], '味わいグループ': new_clusters + 1}))
for i, tag in enumerate(new_tags):
    top3 = np.argsort(cosine_similarity([X_new[i]], X)[0])[::-1][:3]
    print(f'{tag} に近い既存銘柄:', [brand_of(tags[j]) for j in top3])

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
for c in range(N_CLUSTERS):
    pts = X_pca[cluster_labels == c]
    draw_kde_cloud(ax, pts, cluster_colors[c])
    ax.scatter(pts[:, 0], pts[:, 1], alpha=0.35, color=cluster_colors[c], s=40, edgecolors='none', zorder=2, label=f'グループ {c + 1}')
for i, tag in enumerate(new_tags):
    ax.scatter(*X_new_pca[i], color=cluster_colors[new_clusters[i]], marker='*', s=500, edgecolors='black', linewidth=1.8, zorder=10)
    ax.annotate(f'★{brand_of(tag)}', X_new_pca[i], xytext=(10, 10), textcoords='offset points', fontsize=13, weight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7), zorder=11)
ax.set_title('既存モデルへの新規銘柄の照合結果', fontsize=15)
ax.set_xlabel(f'PC1（寄与率 {variance_ratio[0]*100:.2f}%）'); ax.set_ylabel(f'PC2（寄与率 {variance_ratio[1]*100:.2f}%）')
ax.grid(True, linestyle='--', alpha=0.3); ax.legend(loc='upper right')
plt.tight_layout(); plt.show()

## 6. 水の硬度との関係（検証）

「水が硬い/軟らかい蔵元で味わいが違うか」を検証します。
同じ蔵元の複数銘柄は独立ではない（疑似反復）ため、**蔵元ごとにベクトルを平均**してから評価します。

1. マンテル検定：味わいベクトルの距離 と 硬度の差 の相関
2. 交差検証つきRidge回帰：ベクトルから硬度を予測できるか
3. 味わいキーワード別の相関
4. 味わいグループ間の硬度差（ANOVA）

In [ ]:
from scipy.spatial.distance import pdist, squareform
from scipy.stats import pearsonr, spearmanr, f_oneway
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler

N_PERMUTATIONS = 999

hardness = pd.read_csv(WATER_MASTER_PATH).set_index('maker')['water_hardness']
makers = np.array([t.split('_')[2] if len(t.split('_')) == 3 else '不明' for t in tags])

# 蔵元単位に集約（ベクトルを平均）して硬度と突合
maker_vec = pd.DataFrame(X, index=tags).groupby(makers).mean()
merged = maker_vec.join(hardness.rename('hardness'), how='inner').dropna(subset=['hardness'])
X_maker = merged.drop(columns='hardness').values
h = merged['hardness'].values
print(f'銘柄 {len(tags)} → 蔵元 {len(maker_vec)} → 硬度と突合できた蔵元 {len(merged)}')

In [ ]:
def mantel_test(d1, d2, n_perm=999, seed=42):
    """2つの距離行列の相関（Spearman）を並べ替え検定で評価する"""
    idx = np.triu_indices_from(d1, k=1)
    r_obs = spearmanr(d1[idx], d2[idx])[0]
    rng = np.random.default_rng(seed)
    perm = np.array([spearmanr(d1[idx], d2[np.ix_(p, p)][idx])[0]
                     for p in (rng.permutation(len(d1)) for _ in range(n_perm))])
    p_value = (np.sum(np.abs(perm) >= abs(r_obs)) + 1) / (n_perm + 1)
    return r_obs, p_value, perm

vec_dist = squareform(pdist(X_maker, metric='cosine'))
hard_dist = np.abs(h[:, None] - h[None, :])
r, p, perm = mantel_test(vec_dist, hard_dist, N_PERMUTATIONS)
print(f'① マンテル検定: r={r:.4f}, p={p:.4f} →', '有意' if p < 0.05 else '有意な関連なし')

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(perm, bins=40, alpha=0.6, color='gray', label='並べ替え分布（帰無仮説）')
ax.axvline(r, color='red', linewidth=2, label=f'観測値 r={r:.3f}')
ax.set_xlabel('マンテル相関係数'); ax.set_ylabel('頻度'); ax.set_title(f'マンテル検定（p={p:.4f}）'); ax.legend()
plt.tight_layout(); plt.show()

# ② ベクトルから硬度を予測できるか（交差検証R²）
n_splits = min(5, len(X_maker) // 2)
cv_r2 = cross_val_score(Ridge(alpha=10.0), StandardScaler().fit_transform(X_maker), h,
                        cv=KFold(n_splits, shuffle=True, random_state=42), scoring='r2')
print(f'② Ridge回帰 交差検証R² = {cv_r2.mean():.4f} (±{cv_r2.std():.4f})')

In [ ]:
# ③ 味わいキーワード別の相関
maker_taste = taste_scaled.groupby(makers).mean().join(hardness.rename('hardness'), how='inner').dropna(subset=['hardness'])
corr = pd.DataFrame([{'味わい': c, '相関係数': pearsonr(maker_taste[c], maker_taste['hardness'])[0],
                      'p値': pearsonr(maker_taste[c], maker_taste['hardness'])[1]} for c in taste_scaled.columns])
corr = corr.sort_values('相関係数', key=abs, ascending=False)
print(corr.to_string(index=False))

# ④ 味わいグループ間で硬度に差があるか（ANOVA）
df_h = pd.DataFrame({'cluster': cluster_labels, 'hardness': pd.Series(makers).map(hardness).values}).dropna()
f_stat, p_val = f_oneway(*[g['hardness'].values for _, g in df_h.groupby('cluster')])
k = df_h['cluster'].nunique()
eta2 = f_stat * (k - 1) / (f_stat * (k - 1) + len(df_h) - k)
print(f'\n④ ANOVA: F={f_stat:.3f}, p={p_val:.4g}, η²={eta2*100:.2f}%')